In [ ]:
import sys
import os
from pathlib import Path

# current_dir=str(Path(os.getcwd()).parent.parent.parent)
# print(f"Current dir: {current_dir}")
# sys.path.insert(0, current_dir)

if __name__ == "__main__":
    root_path = str(Path(os.getcwd()).parents[1])
    print(f"Root path: {root_path}")
    sys.path.insert(0, root_path)
    if __package__ is None:
        __package__ = "comfyui-image-scorer.external_modules.step01ranking_new"
    print(f"Package: {__package__}")


list chains and group by size

In [ ]:
from shared.graph.crystal_graph import crystal_graph, ChainProxy, NodeProxy
from tqdm import tqdm

chains: list[ChainProxy] = crystal_graph.get_all_chains()

chain_map: dict[int, list[ChainProxy]] = {}
with tqdm(total=len(chains), desc="Mapping chains by length") as pbar:
    for c in chains:
        if c.length not in chain_map:
            chain_map[c.length] = []
        chain_map[c.length].append(c)
        pbar.update(1)

# sort map by length
chain_map = dict(sorted(chain_map.items(), key=lambda item: item[0]))

# print by size
for length, chain_list in chain_map.items():
    print(f"Length {length}: {len(chain_list)} chains")
    print(f" chains (first 5): {chain_list[:5]}")


make a second map with chains as list of nodes, and mark the valid ones (the ones that have the current chain as its main chain)

In [ ]:
all_nodes: list[NodeProxy] = crystal_graph.get_all_nodes()
print(f"Total nodes: {len(all_nodes)}")

main_chains: dict[int, list[str]] = {}
with tqdm(all_nodes, desc="Processing nodes") as pbar:
    for node in all_nodes:
        chain_id: int = node.get_chain()[0].id
        if chain_id not in main_chains:
            main_chains[chain_id] = []
        main_chains[chain_id].append(node.filename)
        pbar.update(1)

print(
    f"sample main chains (5): {[(id,chain[:5]) for id,chain in main_chains.items()][:5]}"
)


create final map

In [ ]:
final_map: dict[int, dict[int, list[tuple[NodeProxy, bool]]]] = {}
# dict [chain_length, dict[chain_id, list[tuple[node, is_main_node]]]]
with tqdm(total=len(chains), desc="Building final map") as pbar:
    for lengths in chain_map:
        for chain in chain_map[lengths]:
            if chain.id not in main_chains:
                print(f"Warning: Chain ID {chain.id} not found in main_chains")
                continue

            if chain.length not in final_map:
                final_map[chain.length] = {}

            local_main_chains: list[str] = main_chains[chain.id]

            final_chain: list[tuple[NodeProxy, bool]] = []
            nodes_in_chain: list[NodeProxy] = chain.get_nodes()

            for node in nodes_in_chain:
                is_main_node: bool = node.filename in local_main_chains
                final_chain.append((node, is_main_node))

            final_map[chain.length][chain.id] = final_chain
            pbar.update(1)

print("\nFinal map (first 5 lengths):")
for length, chains in list(final_map.items())[:5]:
    print(f"Length {length}: {len(chains)} chains")
    for chain_id, nodes in list(chains.items())[:5]:
        print(
            f" Chain ID: {chain_id}, Main nodes: {[node.filename for node, is_main in nodes if is_main]}, All Nodes: {[(node.filename,is_main) for node, is_main in nodes]} "
        )
